In [1]:
import serial
import time
import binascii
import struct

def calculate_crc(data):
    """Расчет CRC16 для EDP"""
    crc = 0xFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            if crc & 0x0001:
                crc >>= 1
                crc ^= 0xA001
            else:
                crc >>= 1
    return crc

def send_hex_command(port, slave_address, function_code, start_address, num_registers, response_length):
    """
    Отправка HEX команды и получение ответа
    
    Args:
        port: порт (например, '/dev/ttyUSB0' или 'COM3')
        slave_address: адрес ведомого устройства
        function_code: код функции (0x03 - чтение, 0x06 - запись, и т.д.)
        start_address: начальный адрес
        num_registers: количество регистров
        response_length: ожидаемая длина ответа в байтах
    """
    
    # Формирование команды
    command = bytearray([
        slave_address,
        function_code,
        (start_address >> 8) & 0xFF,  # старший байт адреса
        start_address & 0xFF,          # младший байт адреса
        (num_registers >> 8) & 0xFF,   # старший байт количества
        num_registers & 0xFF           # младший байт количества
    ])
    
    # Расчет и добавление CRC
    crc = calculate_crc(command)
    command.append(crc & 0xFF)        # младший байт CRC
    command.append((crc >> 8) & 0xFF) # старший байт CRC
    
    print(f"Отправка команды (HEX): {binascii.hexlify(command).upper().decode()}")
    
    try:
        # Открытие порта
        ser = serial.Serial(
            port=port,
            baudrate=9600,
            bytesize=8,
            parity='N',
            stopbits=1,
            timeout=1
        )
        
        # Отправка команды
        ser.write(command)
        time.sleep(0.1)
        
        # Чтение ответа
        response = ser.read(response_length)
        ser.close()
        
        if response:
            print(f"Получен ответ (HEX): {binascii.hexlify(response).upper().decode()}")
            
            # Проверка CRC ответа (если длина позволяет)
            if len(response) >= 2:
                received_crc = (response[-2] | (response[-1] << 8))
                calc_crc = calculate_crc(response[:-2])
                if received_crc == calc_crc:
                    print("CRC корректен")
                else:
                    print("Ошибка CRC!")
            
            return response
        else:
            print("Нет ответа от устройства")
            return None
            
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

# Пример использования
if __name__ == "__main__":
    # Пример для чтения 1 регистра по адресу 0x01
    # Адрес устройства: 0x01, Функция: 0x03, Адрес: 0x0001, Количество: 0x0001
    # Ответ будет содержать: адрес(1) + функция(1) + кол-во байт(1) + данные(2) + CRC(2) = 7 байт
    response = send_hex_command(
        port='COM5',  # или 'COM3' для Windows
        slave_address=0x12,
        function_code=0x03,
        start_address=0x00c9,
        num_registers=0x0004,
        response_length=9
    )


# Парсинг ответа (если есть)
if response and len(response) >= 5:
    # Длина данных находится в 3-м байте (индекс 2)
    data_length = response[2]
    # Данные начинаются с индекса 3
    data_bytes = response[3:3+data_length]
    
    print(f"Сырые данные (HEX): {binascii.hexlify(data_bytes).upper().decode()}")
    print(f"Длина данных: {data_length} байт")
    
    # ============================================
    # 1. Преобразование в целое число (16 бит)
    # ============================================
    if data_length >= 2:
        value_uint16 = (data_bytes[0] << 8) | data_bytes[1]
        print(f"Целое (16 бит): {value_uint16} (0x{value_uint16:04X})")
    
    # ============================================
    # 2. Преобразование в float (32 бита)
    # ============================================
    if data_length >= 4:
        # Берем 4 байта для float
        float_bytes = data_bytes[:4]
        
        print(f"\nFloat данные: {binascii.hexlify(float_bytes).upper().decode()}")
        
        # Вариант 1: Big-Endian (стандартный)
        try:
            float_be = struct.unpack('>f', float_bytes)[0]
            print(f"Float (BE): {float_be:.10f}")
        except:
            pass
        
        # Вариант 2: Little-Endian
        try:
            float_le = struct.unpack('<f', float_bytes)[0]
            print(f"Float (LE): {float_le:.10f}")
        except:
            pass
        
        # Вариант 3: Word-Swapped (часто в Modbus)
        # Меняем местами слова: [байт2, байт3, байт0, байт1]
        try:
            swapped = float_bytes[2:4] + float_bytes[0:2]
            float_sw = struct.unpack('>f', swapped)[0]
            print(f"Float (SW): {float_sw:.10f}")
        except:
            pass
        
        # Вариант 4: Byte-Swapped (полностью обратный порядок)
        try:
            float_bs = struct.unpack('<f', float_bytes[::-1])[0]
            print(f"Float (BS): {float_bs:.10f}")
        except:
            pass
    
    # ============================================
    # 3. Преобразование в 32-битное целое
    # ============================================
    if data_length >= 4:
        int_bytes = data_bytes[:4]
        
        # Unsigned Long (BE)
        try:
            ulong_be = struct.unpack('>I', int_bytes)[0]
            print(f"\nUnsigned Long (BE): {ulong_be} (0x{ulong_be:08X})")
        except:
            pass
        
        # Unsigned Long (LE)
        try:
            ulong_le = struct.unpack('<I', int_bytes)[0]
            print(f"Unsigned Long (LE): {ulong_le} (0x{ulong_le:08X})")
        except:
            pass
        
        # Signed Long (BE)
        try:
            slong_be = struct.unpack('>i', int_bytes)[0]
            print(f"Signed Long (BE): {slong_be} (0x{slong_be:08X})")
        except:
            pass
    
    # ============================================
    # 4. Если данных больше 4 байт - несколько регистров
    # ============================================
    if data_length > 4:
        print(f"\nДанные: {binascii.hexlify(data_bytes).upper().decode()}")
        print("Все регистры:")
        for i in range(0, data_length, 2):
            if i + 1 < data_length:
                reg_value = (data_bytes[i] << 8) | data_bytes[i+1]
                print(f"  Регистр {i//2 + 1}: {reg_value} (0x{reg_value:04X})")

Отправка команды (HEX): 120300C900049694
Нет ответа от устройства


In [1]:
import serial
import time
import binascii
import struct

def calculate_crc(data):
    """Расчет CRC16 для Modbus RTU"""
    crc = 0x0000
    for byte in data:
        crc += byte
    crc -= 0xAA  
    print(crc)
    return crc

def send_hex_command(port, slave_address, function_code, start_address, num_registers, response_length):
    """
    Отправка HEX команды и получение ответа
    
    Args:
        port: порт (например, '/dev/ttyUSB0' или 'COM3')
        slave_address: адрес ведомого устройства
        function_code: код функции (0x03 - чтение, 0x06 - запись, и т.д.)
        start_address: начальный адрес
        num_registers: количество регистров
        response_length: ожидаемая длина ответа в байтах
    """
    
    # Формирование команды
    command = bytearray([0x55, 0x55, 0x12, 0x01, 0x84, 0x05, 0x00, 0x08, 0x00, 0x00, 0x12, 0x00])  
    command1 = bytearray([     
        0x55,
        0x55,
        slave_address,                 #DST адрес назначения 55 55 12 03 84 05 00 08 00 01 02 00 A9
        0x01,                          #SRC адрес отправителя
        0x01,                          #CNT счетчик пакетов
        num_registers & 0xFF,           # младший байт длины данных
        (num_registers >> 8) & 0xFF,   # старший байт длины данных
        function_code,
        start_address & 0xFF,          # младший байт адреса
        (start_address >> 8) & 0xFF,  # старший байт адреса
        num_registers & 0xFF,           # младший байт длины принимаемых данных
        (num_registers >> 8) & 0xFF,   # старший байт длины принимаемых данных
        
        
    ])
    
    # Расчет и добавление CRC
    crc = calculate_crc(command)    
    command.append(crc & 0xFF)        # младший байт CRC
    #command.append((crc >> 8) & 0xFF) # старший байт CRC
    
    print(f"Отправка команды (HEX): {binascii.hexlify(command).upper().decode()}")
    
    try:
        # Открытие порта
        ser = serial.Serial(
            port=port,
            baudrate=57600, #9600,
            bytesize=8,
            parity='N',
            stopbits=1,
            timeout=1
        )
        
        # Отправка команды
        ser.write(command)
        time.sleep(0.1)
        
        # Чтение ответа
        real_length = response_length + 16
        response = ser.read(real_length)
        ser.close()
        
        if response:
            print(f"Получен ответ (HEX): {binascii.hexlify(response).upper().decode()}")
            
            # Проверка CRC ответа (если длина позволяет)
            if len(response) >= 2:
                received_crc = ((response[-1]))
                calc_crc = calculate_crc(response[:-1])
                if received_crc == calc_crc:
                    print("CRC корректен")
                else:
                    print("Ошибка CRC!")
            
            return response
        else:
            print("Нет ответа от устройства")
            return None
            
    except Exception as e:
        print(f"Ошибка: {e}")
        return None

# Пример использования
if __name__ == "__main__":
    # Пример для чтения 1 регистра по адресу 0x01
    # Адрес устройства: 0x01, Функция: 0x03, Адрес: 0x0001, Количество: 0x0001
    # Ответ будет содержать: адрес(1) + функция(1) + кол-во байт(1) + данные(2) + CRC(2) = 7 байт
    response = send_hex_command(
        port='COM5',  # или 'COM3' для Windows
        slave_address=0x12,
        function_code=0x06,
        start_address=0x2004,
        num_registers=0x0002,
        response_length=9
    )


# Парсинг ответа (если есть)
if response and len(response) >= 5:
    # Длина данных находится в 3-м байте (индекс 2)
    data_length = response[2]
    # Данные начинаются с индекса 3
    data_bytes = response[3:3+data_length]
    
    print(f"Сырые данные (HEX): {binascii.hexlify(data_bytes).upper().decode()}")
    print(f"Длина данных: {data_length} байт")
    
    # ============================================
    # 1. Преобразование в целое число (16 бит)
    # ============================================
    if data_length >= 2:
        value_uint16 = (data_bytes[0] << 8) | data_bytes[1]
        print(f"Целое (16 бит): {value_uint16} (0x{value_uint16:04X})")
    
    # ============================================
    # 2. Преобразование в float (32 бита)
    # ============================================
    if data_length >= 4:
        # Берем 4 байта для float
        float_bytes = data_bytes[:4]
        
        print(f"\nFloat данные: {binascii.hexlify(float_bytes).upper().decode()}")
        
        # Вариант 1: Big-Endian (стандартный)
        try:
            float_be = struct.unpack('>f', float_bytes)[0]
            print(f"Float (BE): {float_be:.10f}")
        except:
            pass
        
        # Вариант 2: Little-Endian
        try:
            float_le = struct.unpack('<f', float_bytes)[0]
            print(f"Float (LE): {float_le:.10f}")
        except:
            pass
        
        # Вариант 3: Word-Swapped (часто в Modbus)
        # Меняем местами слова: [байт2, байт3, байт0, байт1]
        try:
            swapped = float_bytes[2:4] + float_bytes[0:2]
            float_sw = struct.unpack('>f', swapped)[0]
            print(f"Float (SW): {float_sw:.10f}")
        except:
            pass
        
        # Вариант 4: Byte-Swapped (полностью обратный порядок)
        try:
            float_bs = struct.unpack('<f', float_bytes[::-1])[0]
            print(f"Float (BS): {float_bs:.10f}")
        except:
            pass
    
    # ============================================
    # 3. Преобразование в 32-битное целое
    # ============================================
    if data_length >= 4:
        int_bytes = data_bytes[:4]
        
        # Unsigned Long (BE)
        try:
            ulong_be = struct.unpack('>I', int_bytes)[0]
            print(f"\nUnsigned Long (BE): {ulong_be} (0x{ulong_be:08X})")
        except:
            pass
        
        # Unsigned Long (LE)
        try:
            ulong_le = struct.unpack('<I', int_bytes)[0]
            print(f"Unsigned Long (LE): {ulong_le} (0x{ulong_le:08X})")
        except:
            pass
        
        # Signed Long (BE)
        try:
            slong_be = struct.unpack('>i', int_bytes)[0]
            print(f"Signed Long (BE): {slong_be} (0x{slong_be:08X})")
        except:
            pass
    
    # ============================================
    # 4. Если данных больше 4 байт - несколько регистров
    # ============================================
    if data_length > 4:
        print(f"\nДанные: {binascii.hexlify(data_bytes).upper().decode()}")
        print("Все регистры:")
        for i in range(0, data_length, 2):
            if i + 1 < data_length:
                reg_value = (data_bytes[i] << 8) | data_bytes[i+1]
                print(f"  Регистр {i//2 + 1}: {reg_value} (0x{reg_value:04X})")

182
Отправка команды (HEX): 555512018405000800001200B6
Получен ответ (HEX): 5555011284120006000F000200003D00000000000000000000
253
Ошибка CRC!
Сырые данные (HEX): 12
Длина данных: 1 байт


In [4]:
import serial
import binascii
import time

class RawModbusTerminal:
    def __init__(self, port, baudrate=9600):
        self.port = port
        self.baudrate = baudrate
        self.ser = None
    
    def connect(self):
        try:
            self.ser = serial.Serial(self.port, self.baudrate, timeout=2)
            print(f"✓ Подключен к {self.port}")
            return True
        except Exception as e:
            print(f"✗ Ошибка: {e}")
            return False
    
    def send_raw_hex(self, hex_string):
        """Отправка HEX команды"""
        # Удаляем пробелы и конвертируем
        hex_string = hex_string.replace(' ', '')
        data = bytes.fromhex(hex_string)
        
        print(f"\n➜ Отправка: {binascii.hexlify(data).upper().decode()}")
        self.ser.write(data)
        time.sleep(0.05)
    
    def read_raw(self, timeout=1):
        """Чтение сырых данных"""
        # Даем время на ответ
        time.sleep(timeout)
        
        # Читаем все доступные данные
        available = self.ser.in_waiting
        if available == 0:
            print("✗ Нет данных в буфере")
            return None
        
        print(f"✓ Доступно байт: {available}")
        
        # Читаем все
        raw = self.ser.read(available)
        
        # Вывод в разных форматах
        print(f"\n📊 Сырые данные:")
        print(f"  HEX: {binascii.hexlify(raw).upper().decode()}")
        print(f"  Длина: {len(raw)} байт")
        print(f"  DEC: {[b for b in raw]}")
        print(f"  ASCII: {repr(raw)}")
        
        # Побайтово с адресами
        print("\n  Побайтово:")
        for i, b in enumerate(raw):
            print(f"    [{i:02d}] 0x{b:02X} ({b:3d})")
        
        return raw
    
    def interactive(self):
        """Интерактивный режим"""
        print("\n" + "="*50)
        print("RAW MODBUS TERMINAL")
        print("="*50)
        print("Введите HEX команду (без пробелов или с пробелами)")
        print("Пример: 12 03 01 39 00 01 D7 69")
        print("Команды:")
        print("  'exit' - выход")
        print("  'clear' - очистить буфер")
        print("  'read' - прочитать без отправки")
        print("="*50 + "\n")
        
        while True:
            cmd = input("\nHEX> ").strip()
            
            if cmd.lower() == 'exit':
                break
            elif cmd.lower() == 'clear':
                self.ser.reset_input_buffer()
                self.ser.reset_output_buffer()
                print("✓ Буфер очищен")
                continue
            elif cmd.lower() == 'read':
                self.read_raw()
                continue
            
            # Отправка команды
            try:
                self.send_raw_hex(cmd)
                self.read_raw()
            except Exception as e:
                print(f"✗ Ошибка: {e}")
    
    def close(self):
        if self.ser:
            self.ser.close()
            print("✓ Порт закрыт")

# Использование
terminal = RawModbusTerminal('COM5', 9600)
if terminal.connect():
    terminal.interactive()
    terminal.close()

✓ Подключен к COM5

RAW MODBUS TERMINAL
Введите HEX команду (без пробелов или с пробелами)
Пример: 12 03 01 39 00 01 D7 69
Команды:
  'exit' - выход
  'clear' - очистить буфер
  'read' - прочитать без отправки




HEX>  120301390001d769



➜ Отправка: 120301390001D769
✓ Доступно байт: Serial<id=0x1c9039ab6d0, open=True>(port='COM5', baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=2, xonxoff=False, rtscts=False, dsrdtr=False)
✗ Ошибка: '>' not supported between instances of 'Serial' and 'int'



HEX>  exit


✓ Порт закрыт


In [2]:
import serial, binascii, time

# Одна строка для отправки и чтения
ser = serial.Serial('COM5', 9600, timeout=2)
ser.write(bytes.fromhex('120301390001D769'))
time.sleep(0.1)
data = ser.read(ser.in_waiting or 256)
ser.close()
print(binascii.hexlify(data).upper().decode() if data else "Нет ответа")

Нет ответа


In [21]:
import serial, binascii, time
class ModbusReader:
    def __init__(self, port, baudrate=9600):
        self.port = port
        self.baudrate = baudrate
    
    def read(self, hex_command, extra_bytes=0):
        """
        Чтение данных с возможностью получить лишние байты
        
        Args:
            hex_command: HEX команда
            extra_bytes: сколько дополнительных байт прочитать (0, 1, 2, ...)
        """
        try:
            ser = serial.Serial(self.port, self.baudrate, timeout=2)
            
            # Отправка
            cmd = bytes.fromhex(hex_command.replace(' ', ''))
            ser.write(cmd)
            time.sleep(0.1)
            
            # Определяем сколько читать
            if ser.in_waiting > 0:
                # Читаем все доступные + extra
                base_bytes = ser.in_waiting
                total_to_read = base_bytes + extra_bytes
                response = ser.read(total_to_read)
                
                print(f"Базовых байт: {base_bytes}")
                print(f"Дополнительных: {extra_bytes}")
                print(f"Всего прочитано: {len(response)}")
                
                if response:
                    print(f"HEX: {binascii.hexlify(response).upper().decode()}")
                
                ser.close()
                return response
            else:
                ser.close()
                print("Нет данных")
                return None
                
        except Exception as e:
            print(f"Ошибка: {e}")
            return None

# Использование
reader = ModbusReader('COM5')
response = reader.read('120300C90001D769', extra_bytes=4)  # Читаем на 1 байт больше

Нет данных


In [30]:
import serial
import binascii
import time

# Демонстрация пошагового расчета
def calculate_crc_step_by_step(data):
    crc = 0xFFFF
    print(f"Начальное CRC: {crc:04X}")
    print("-" * 50)
    
    for byte_index, byte in enumerate(data):
        print(f"\nБайт {byte_index}: 0x{byte:02X}")
        crc ^= byte
        print(f"  После XOR: {crc:04X}")
        
        for bit in range(8):
            if crc & 0x0001:
                crc >>= 1
                crc ^= 0xA001
                print(f"    Бит {bit}: 1 -> сдвиг и XOR: {crc:04X}")
            else:
                crc >>= 1
                print(f"    Бит {bit}: 0 -> сдвиг: {crc:04X}")
        
        print(f"  CRC после байта: {crc:04X}")
    
    return crc

# Рассчитаем для команды
command_without_crc = bytes([0x12, 0x03, 0x00, 0x01, 0x00, 0x02])
crc = calculate_crc_step_by_step(command_without_crc)
print(f"\n✅ Итоговый CRC: {crc:04X}")
print(f"   Младший байт: {crc & 0xFF:02X}")
print(f"   Старший байт: {(crc >> 8) & 0xFF:02X}")

Начальное CRC: FFFF
--------------------------------------------------

Байт 0: 0x12
  После XOR: FFED
    Бит 0: 1 -> сдвиг и XOR: DFF7
    Бит 1: 1 -> сдвиг и XOR: CFFA
    Бит 2: 0 -> сдвиг: 67FD
    Бит 3: 1 -> сдвиг и XOR: 93FF
    Бит 4: 1 -> сдвиг и XOR: E9FE
    Бит 5: 0 -> сдвиг: 74FF
    Бит 6: 1 -> сдвиг и XOR: 9A7E
    Бит 7: 0 -> сдвиг: 4D3F
  CRC после байта: 4D3F

Байт 1: 0x03
  После XOR: 4D3C
    Бит 0: 0 -> сдвиг: 269E
    Бит 1: 0 -> сдвиг: 134F
    Бит 2: 1 -> сдвиг и XOR: A9A6
    Бит 3: 0 -> сдвиг: 54D3
    Бит 4: 1 -> сдвиг и XOR: 8A68
    Бит 5: 0 -> сдвиг: 4534
    Бит 6: 0 -> сдвиг: 229A
    Бит 7: 0 -> сдвиг: 114D
  CRC после байта: 114D

Байт 2: 0x00
  После XOR: 114D
    Бит 0: 1 -> сдвиг и XOR: A8A7
    Бит 1: 1 -> сдвиг и XOR: F452
    Бит 2: 0 -> сдвиг: 7A29
    Бит 3: 1 -> сдвиг и XOR: 9D15
    Бит 4: 1 -> сдвиг и XOR: EE8B
    Бит 5: 1 -> сдвиг и XOR: D744
    Бит 6: 0 -> сдвиг: 6BA2
    Бит 7: 0 -> сдвиг: 35D1
  CRC после байта: 35D1

Байт 3: 0x01
  П